In [23]:
import numpy as np
import pandas as pd
import scipy.stats as stats

In [24]:
!bcftools query -f '[%GT\t]\n' ../1000G_chr22_pruned.vcf.gz > ../chr22_genotypes.txt

In [25]:
!bcftools query -f '%ID\n' ../1000G_chr22_pruned.vcf.gz > ../chr22_snp_names.txt

In [26]:
snp_names = pd.read_csv('../chr22_snp_names.txt', header=None)[0].tolist()

In [27]:
def load_vcf_genotypes(filename):
    df = pd.read_csv(filename, sep='\t', header=None, low_memory=False)
    
    #function to convert '0|1' or '1/1' to an integer (0, 1, or 2)
    def parse_gt(gt_string):
        if pd.isna(gt_string): return 0
        # Replace | with / to handle both phased and unphased
        alleles = str(gt_string).replace('|', '/').split('/')
        try:
            return int(alleles[0]) + int(alleles[1])
        except (ValueError, IndexError):
            return 0

    #apply the converter to all columns except the last one if it's empty
    genotypes = df.iloc[:, :-1].applymap(parse_gt).values
    return genotypes

In [28]:
G = load_vcf_genotypes('../chr22_genotypes.txt')
print(f"Loaded genotype matrix with shape: {G.shape}")

/var/folders/y5/f6l5v6h503sdy7kc_3d__qjh0000gn/T/ipykernel_68282/1707456386.py:15: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  genotypes = df.iloc[:, :-1].applymap(parse_gt).values


Loaded genotype matrix with shape: (13629, 2504)


In [29]:
variances = np.var(G, axis=1)
valid_snp_mask = variances > 0

# 2. Filter both your genotype matrix and your SNP names list
G_filtered = G[valid_snp_mask]
snp_names_filtered = np.array(snp_names)[valid_snp_mask]

print(f"Removed {len(G) - len(G_filtered)} invariant SNPs.")

Removed 4 invariant SNPs.


In [31]:
# 3. Calculate LD Scores on the filtered matrix
# np.corrcoef expects variables in rows, which G_filtered already is
corr_matrix = np.corrcoef(G_filtered)

# Handle any remaining potential NaNs just in case
corr_matrix = np.nan_to_num(corr_matrix)

# Square the correlations and sum them to get LD scores
ld_scores = np.sum(corr_matrix**2, axis=1)

# 4. Save the results
ld_df = pd.DataFrame({
    'SNP': snp_names_filtered,
    'LD_SCORE': ld_scores
})
ld_df.to_csv('sim_ldscores.csv', index=False)
print("LD scores calculated and saved successfully.")

LD scores calculated and saved successfully.


In [32]:
num_snps, num_samples = G_filtered.shape

benchmarking scenario 1: polygenic architecture

In [33]:
#select 5% of SNPs to be causal
perc_causal = 0.05
causal_idx = np.random.choice(num_snps, int(num_snps * perc_causal), replace=False)

#assign effect sizes (beta) and generate phenotype Y
true_betas = np.zeros(num_snps)
true_betas[causal_idx] = np.random.normal(0, 1, len(causal_idx))
genetic_signal = np.dot(G_filtered.T, true_betas)
noise = np.random.normal(0, np.std(genetic_signal), num_samples) # h2 ~ 0.5
Y_polygenic = genetic_signal + noise

benchmarking scenario 2: pure inflation (confounding)

In [34]:
#no genetic signal->all betas = 0, only random noise plus a bias
Y_inflated = np.random.normal(0, 1, num_samples)
inflation_shift = 0.5 #simulates population stratification/bias

In [35]:
def run_mini_gwas(phenotype, genotypes):
    results = []
    for i in range(num_snps):
        snp = genotypes[i, :]
        slope, intercept, r_val, p_val, std_err = stats.linregress(snp, phenotype)
        results.append(['rs_sim_' + str(i), slope, std_err, p_val])
    return pd.DataFrame(results, columns=['MarkerName', 'Beta', 'SE', 'Pvalue'])

In [36]:
df_poly = run_mini_gwas(Y_polygenic, G)
df_poly.to_csv('sim_polygenic.tsv', sep='\t', index=False)

df_infl = run_mini_gwas(Y_inflated, G)
#manually inflate chi-sq statistics for inflated scenario
df_infl['Pvalue'] = df_infl['Pvalue'] * 0.1 #simplistic way to simulate bias
df_infl.to_csv('sim_inflated.tsv', sep='\t', index=False)

ValueError: Cannot calculate a linear regression if all x values are identical

In [ ]:
df_poly

In [ ]:
df_infl

In [37]:
import numpy as np
import pandas as pd

# 1. Identify SNPs with variance > 0
# Row-wise variance (axis=1 because SNPs are rows)
variances = np.var(G, axis=1)
valid_snp_mask = variances > 0

# 2. Filter both your genotype matrix and your SNP names list
G_filtered = G[valid_snp_mask]
snp_names_filtered = np.array(snp_names)[valid_snp_mask]

print(f"Removed {len(G) - len(G_filtered)} invariant SNPs.")

# 3. Calculate LD Scores on the filtered matrix
# np.corrcoef expects variables in rows, which G_filtered already is
corr_matrix = np.corrcoef(G_filtered)

# Handle any remaining potential NaNs just in case
corr_matrix = np.nan_to_num(corr_matrix)

# Square the correlations and sum them to get LD scores
ld_scores = np.sum(corr_matrix**2, axis=1)

# 4. Save the results
ld_df = pd.DataFrame({
    'SNP': snp_names_filtered,
    'LD_SCORE': ld_scores
})
ld_df.to_csv('sim_ldscores.csv', index=False)
print("LD scores calculated and saved successfully.")

Removed 4 invariant SNPs.
LD scores calculated and saved successfully.


In [38]:
ld_df

,SNP,LD_SCORE
0,rs4911642,129.093607
1,rs5747010,80.750751
2,rs7286836,32.059005
3,rs9604821,48.219934
4,rs9605903,118.450135
...,...,...
13620,rs3810648,233.329395
13621,rs2285395,78.564757
13622,rs13056621,215.304441
13623,rs5771007,28.080258


In [40]:
import pandas as pd

# 1. Transform the Simulated Summary Stats
for filename in ['./simulated_data/sim_polygenic.tsv', './simulated_data/sim_inflated.tsv']:
    df = pd.read_csv(filename, sep='\t')
    df['CHR'] = 22  # Since we used Chromosome 22 genotypes
    df = df.rename(columns={'MarkerName': 'SNP'})
    df.to_csv(filename, sep='\t', index=False)

# 2. Transform your LD Scores
# Your script expects a column named 'L2' and tab-separation
ld_df = pd.read_csv('sim_ldscores.csv') # Based on our previous step
ld_df['CHR'] = 22
ld_df = ld_df.rename(columns={'LD_SCORE': 'L2'})
ld_df.to_csv('sim_ldscores.txt', sep='\t', index=False)